# Prepare ontology evidence

Run with `RDFSOLVE_NOTEBOOK=03_ontology_cache.ipynb RDFSOLVE_RUN_KIND=ontology-cache sbatch scripts/slurm_mcp_mine.sh`.

Keep the source snapshot and external evidence separate. This notebook sees schema IRIs and question concepts; reference answers are computed only in the evaluation notebook. Reuse existing snapshots and archive the cache before refreshing it.

Fresh snapshots retain the configured named graph and explicitly exclude IRIs without a scheme. Existing snapshots are reused; archive them before refreshing.

The AOP sample includes eight human-matching roots and eight roots without that match. Set `RDFSOLVE_REFRESH_SNAPSHOT=1` to rebuild it.


In [ ]:
import json, logging, os
from pathlib import Path
from IPython.display import Markdown, display
from rdfsolve.api import Client
from rdfsolve import SchemaMiner
from rdfsolve.ontology import OntologyLookup, term_key
from rdfsolve.catalogue import Catalogue

root = Path(os.environ["RDFSOLVE_ROOT"])
folder = root / "notebooks/mcp/schemas"
out = Path(os.environ["RDFSOLVE_OUTPUT"])
cache = Path(os.getenv("RDFSOLVE_ONTOLOGY_CACHE", root / "notebooks/mcp/ontology-cache.json"))
lookup = OntologyLookup(cache=cache, max_requests=300)
logging.basicConfig(filename=out / "ontology.log", level=logging.INFO, force=True)


In [ ]:
name = "aopwikirdf-human"
if os.getenv("RDFSOLVE_REFRESH_SNAPSHOT") == "1" or not (folder / f"{name}.ttl").exists():
    with Client.open(folder / "aopwikirdf.schema.json", ontology_grounding=lookup, timeout=60) as live:
        humans = live.find("human", kind="http://purl.bioontology.org/ontology/NCBITAXON/131567")
        assert humans, "No verified human taxon was found in the source"
        identities = " ".join("<" + str(r.uri) + ">" for r in humans)
        assert len(live.graph_uris) == 1, "Choose one named graph for this development snapshot"
        scope = live.graph_uris[0]
        query = """CONSTRUCT { ?s ?p ?o . ?o ?q ?v . ?v a ?type . } WHERE { GRAPH <%s> {
          { { SELECT DISTINCT ?s WHERE { ?s a <http://aopkb.org/aop_ontology#AdverseOutcomePathway> ; <http://purl.bioontology.org/ontology/NCBITAXON/131567> ?human . VALUES ?human { %s } } ORDER BY ?s LIMIT 8 }
            UNION
            { SELECT DISTINCT ?s WHERE { ?s a <http://aopkb.org/aop_ontology#AdverseOutcomePathway> . FILTER NOT EXISTS { ?s <http://purl.bioontology.org/ontology/NCBITAXON/131567> ?human . VALUES ?human { %s } } } ORDER BY ?s LIMIT 8 } }
          ?s ?p ?o . FILTER(!isIRI(?o) || REGEX(STR(?o), "^[A-Za-z][A-Za-z0-9+.-]*:"))
          OPTIONAL { FILTER(isIRI(?o)) ?o ?q ?v . FILTER(!isIRI(?v) || REGEX(STR(?v), "^[A-Za-z][A-Za-z0-9+.-]*:")) OPTIONAL { ?v a ?type } }
        } }""" % (scope, identities, identities)
        graph = live.source.construct_graph(query)
        live.save_session(out / "human-grounding.json")
    graph.serialize(destination=folder / f"{name}.ttl", format="turtle")
    with SchemaMiner.from_graph(graph, counts=False, strategy="one-shot", delay=0, enrich=True) as miner:
        schema = miner.mine(name)
        schema.about.description = "Eight human-matching and eight nonmatching AOP roots in one named graph; two hops; IRIs without a scheme are excluded."
    (folder / f"{name}.schema.json").write_text(json.dumps(schema.to_dict(), indent=2))
    (folder / f"{name}.source.rq").write_text(query)
    lookup = OntologyLookup(cache=cache, max_requests=300)


In [ ]:
cases = json.loads((root / "notebooks/mcp/ontology-cases.json").read_text())
for source in sorted({case["source"] for case in cases}):
    with Client.open(folder / f"{source}.schema.json", data_file=folder / f"{source}.ttl", ontology_grounding=lookup) as client:
        catalogue = Catalogue(client)
        for iri in sorted(catalogue.known_iris):
            if term_key(iri) != iri:
                client.vocabulary(iri)
        for concept in sorted({c for case in cases if case["source"] == source for c in case["concepts"]}):
            for candidate in lookup.search(concept):
                lookup.lookup(candidate["iri"])
        display(client.describe("measurement method"))
        client.save_session(out / f"{source}-vocabulary.json")
    lookup = OntologyLookup(cache=cache, max_requests=300)
with Client.open(folder / "aopwikirdf-human.schema.json", data_file=folder / "aopwikirdf-human.ttl") as client:
    display(Markdown(client.diagram("Adverse Outcome Pathway", "Key Event")))
print("Frozen evidence:", cache)
